In [1]:
%pip install pandas numpy scikit-learn matplotlib joblib


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
df = pd.read_csv("Dataset/assistments_2009_2010.csv", low_memory=False)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (1011079, 20)


,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response_time,tutor_mode,answer_type,sequence_id,student_class_id,position,problem_set_type,base_sequence_id,list_skill_ids,list_skills,teacher_id,school_id
0,20224085,232368,73963,42904,76429,0,0.0,3,106016.0,tutor,choose_1,6272.0,11816.0,93.0,MasterySection,6272.0,NaN,NaN,22763.0,73.0
1,20224095,232368,73963,42904,76430,0,1.0,1,194187.0,tutor,choose_1,6272.0,11816.0,93.0,MasterySection,6272.0,NaN,NaN,22763.0,73.0
2,20224113,232368,73963,42904,76431,0,1.0,1,12734.0,tutor,algebra,6272.0,11816.0,93.0,MasterySection,6272.0,NaN,NaN,22763.0,73.0
3,20224123,232368,73963,42904,76432,0,1.0,1,333484.0,tutor,choose_1,6272.0,11816.0,93.0,MasterySection,6272.0,NaN,NaN,22763.0,73.0
4,20224142,232368,73963,42904,76433,0,0.0,2,52828.0,tutor,algebra,6272.0,11816.0,93.0,MasterySection,6272.0,NaN,NaN,22763.0,73.0


In [3]:
print("Pandas version:", pd.__version__)
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDuplicate column names:")
print(df.columns[df.columns.duplicated()].tolist())

print("\nNumber of order_id columns:")
print((df.columns == "order_id").sum())

print("\norder_id data type:")
print(df["order_id"].dtype)

print("\nFirst 10 order_id values:")
display(df["order_id"].head(10))

print("\nSample raw values:")
print(df["order_id"].astype(str).head(10).tolist())

Pandas version: 2.3.3
Dataset shape: (1011079, 20)

Column names:
['order_id', 'assignment_id', 'user_id', 'assistment_id', 'problem_id', 'original', 'correct', 'attempt_count', 'ms_first_response_time', 'tutor_mode', 'answer_type', 'sequence_id', 'student_class_id', 'position', 'problem_set_type', 'base_sequence_id', 'list_skill_ids', 'list_skills', 'teacher_id', 'school_id']

Duplicate column names:
[]

Number of order_id columns:
1

order_id data type:
int64

First 10 order_id values:


0    20224085
1    20224095
2    20224113
3    20224123
4    20224142
5    20224159
6    20224180
7    20224183
8    20224187
9    20224190
Name: order_id, dtype: int64


Sample raw values:
['20224085', '20224095', '20224113', '20224123', '20224142', '20224159', '20224180', '20224183', '20224187', '20224190']


In [4]:
# Standardize column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Original dataset shape:", df.shape)
print("Available columns:", df.columns.tolist())

# Check required columns
required_columns = ["user_id", "correct"]

for column in required_columns:
    if column not in df.columns:
        raise ValueError(f"Required column missing: {column}")

# Inspect target values before cleaning
print("\nCorrect values before cleaning:")
print(df["correct"].value_counts(dropna=False).head(10))

# Remove rows with missing student IDs or targets
df = df.dropna(subset=["user_id", "correct"]).copy()

# Convert target to numeric
df["correct"] = pd.to_numeric(df["correct"], errors="coerce")

# Keep valid binary labels
df = df[df["correct"].isin([0, 1])].copy()
df["correct"] = df["correct"].astype(int)

print("\nRows after filtering:", len(df))

# Stop with a useful message if no records remain
if df.empty:
    raise ValueError(
        "No valid rows remain after cleaning. "
        "Check the CSV format and the values in the 'correct' column."
    )

# Sort only when order_id contains valid numeric values
if "order_id" in df.columns:
    order_ids = pd.to_numeric(df["order_id"], errors="coerce")

    # Convert to a regular NumPy array for sorting
    order_array = order_ids.to_numpy(dtype="float64", na_value=np.nan)

    # Sort valid IDs first, with missing IDs at the end
    sort_indices = np.argsort(
        np.where(np.isnan(order_array), np.inf, order_array),
        kind="stable"
    )

    df = df.iloc[sort_indices].copy()

df = df.reset_index(drop=True)

print("Sorting step completed.")
display(df.head())

Original dataset shape: (1011079, 20)
Available columns: ['order_id', 'assignment_id', 'user_id', 'assistment_id', 'problem_id', 'original', 'correct', 'attempt_count', 'ms_first_response_time', 'tutor_mode', 'answer_type', 'sequence_id', 'student_class_id', 'position', 'problem_set_type', 'base_sequence_id', 'list_skill_ids', 'list_skills', 'teacher_id', 'school_id']

Correct values before cleaning:
correct
1.000    635882
0.000    374431
0.750       289
0.500       206
0.250       163
0.875        71
0.625        28
0.375         7
0.125         1
0.850         1
Name: count, dtype: int64

Rows after filtering: 1010313
Sorting step completed.


,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response_time,tutor_mode,answer_type,sequence_id,student_class_id,position,problem_set_type,base_sequence_id,list_skill_ids,list_skills,teacher_id,school_id
0,20224085,232368,73963,42904,76429,0,0,3,106016.0,tutor,choose_1,6272.0,11816.0,93.0,MasterySection,6272.0,NaN,NaN,22763.0,73.0
1,20224095,232368,73963,42904,76430,0,1,1,194187.0,tutor,choose_1,6272.0,11816.0,93.0,MasterySection,6272.0,NaN,NaN,22763.0,73.0
2,20224113,232368,73963,42904,76431,0,1,1,12734.0,tutor,algebra,6272.0,11816.0,93.0,MasterySection,6272.0,NaN,NaN,22763.0,73.0
3,20224123,232368,73963,42904,76432,0,1,1,333484.0,tutor,choose_1,6272.0,11816.0,93.0,MasterySection,6272.0,NaN,NaN,22763.0,73.0
4,20224142,232368,73963,42904,76433,0,0,2,52828.0,tutor,algebra,6272.0,11816.0,93.0,MasterySection,6272.0,NaN,NaN,22763.0,73.0


In [7]:
# Reset the index to ensure it is unique and consecutive
df = df.reset_index(drop=True)

# Normalize student IDs to avoid inconsistent group keys
df["user_id"] = df["user_id"].astype(str)

# Create a positional index
positions = np.arange(len(df))

# Store the number of earlier interactions for each student
prior_attempts = np.zeros(len(df), dtype=np.int64)
prior_correct = np.zeros(len(df), dtype=np.int64)

# Factorize student IDs into integer group codes
student_codes, _ = pd.factorize(df["user_id"], sort=False)

# Process each student's rows in the current dataset order
for code in np.unique(student_codes):
    student_positions = positions[student_codes == code]

    # Number of earlier interactions
    prior_attempts[student_positions] = np.arange(len(student_positions))

    # Correctness values for this student's interactions
    correct_values = df.iloc[student_positions]["correct"].to_numpy(dtype=np.int64)

    # Cumulative correctness excluding the current interaction
    prior_correct[student_positions] = (
        np.cumsum(correct_values) - correct_values
    )

df["prior_attempts"] = prior_attempts
df["prior_correct"] = prior_correct

# Calculate prior accuracy
df["prior_accuracy"] = (
    df["prior_correct"]
    / df["prior_attempts"].replace(0, np.nan)
)

display(
    df[[
        "user_id",
        "correct",
        "prior_attempts",
        "prior_correct",
        "prior_accuracy"
    ]].head(15)
)

,user_id,correct,prior_attempts,prior_correct,prior_accuracy
0,73963,0,0,0,NaN
1,73963,1,1,0,0.000000
2,73963,1,2,1,0.500000
3,73963,1,3,2,0.666667
4,73963,0,4,3,0.750000
5,73963,0,5,3,0.600000
6,73963,0,6,3,0.500000
7,73963,0,7,3,0.428571
8,73963,1,8,3,0.375000
9,73963,1,9,4,0.444444


In [8]:
# Target: what we want the model to predict
target = "correct"

# Numerical features
numeric_features = [
    "prior_attempts",
    "prior_accuracy"
]

# Add original if it exists in the dataset
if "original" in df.columns:
    numeric_features.append("original")

# Categorical features
categorical_features = [
    col for col in [
        "answer_type",
        "tutor_mode",
        "problem_set_type"
    ]
    if col in df.columns
]

# Keep only features that exist
numeric_features = [c for c in numeric_features if c in df.columns]

feature_columns = numeric_features + categorical_features

X = df[feature_columns].copy()
y = df[target].astype(int)

print("Features:", feature_columns)
print("Target distribution:")
print(y.value_counts())

Features: ['prior_attempts', 'prior_accuracy', 'original', 'answer_type', 'tutor_mode', 'problem_set_type']
Target distribution:
correct
1    635882
0    374431
Name: count, dtype: int64


In [9]:
# IMPORTANT:
# Ensure df is sorted chronologically before this cell.
# If order_id represents interaction order:
# df = df.sort_values("order_id").reset_index(drop=True)

split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 808250
Testing rows: 202063


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

print("Pipeline created successfully.")

Pipeline created successfully.


In [11]:
model.fit(X_train, y_train)

print("Model training completed!")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_linea

Model training completed!


In [14]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
import numpy as np

# 1. Generate predictions
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))



Accuracy: 0.7167912977635688

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.25      0.36     63735
           1       0.73      0.93      0.82    138328

    accuracy                           0.72    202063
   macro avg       0.68      0.59      0.59    202063
weighted avg       0.70      0.72      0.67    202063


Confusion Matrix:
[[ 16083  47652]
 [  9574 128754]]


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
